In [1]:
import os
import sys
import pandas as pd
import re
from collections import Counter

# 환경 설정
project_dir = "/data/ephemeral/home/nlp-5/eunbyul/joe"
sys.path.append(project_dir)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# 데이터 로드
train_df = pd.read_csv(os.path.join(project_dir, 'data', 'train.csv'))
val_df = pd.read_csv(os.path.join(project_dir, 'data', 'dev.csv'))
test_df = pd.read_csv(os.path.join(project_dir, 'data', 'test.csv'))

# 전체를 하나의 dict로 관리
datasets = {'train': train_df, 'dev': val_df, 'test': test_df}

In [2]:
def basic_eda(df):
    print("샘플 개수:", len(df))
    print("\n[요약문 길이 통계]")
    if 'summary' in df.columns:
        df['summary_len'] = df['summary'].astype(str).apply(len)
        print(df['summary_len'].describe())
        print(df['summary'].head(5))
        print(df['summary'].tail(5))
    print("\n[dialogue 길이 통계]")
    df['dialogue_len'] = df['dialogue'].astype(str).apply(len)
    print(df['dialogue_len'].describe())
    print(df['dialogue'].head(2))
    print("\n[특수 토큰 빈도]")
    for token in ['#Person1#','#Person2#','#PhoneNumber#','#Address#']:
        print(f"{token} 빈도: {df['dialogue'].astype(str).str.count(token).sum()}")

    # 기타 자소, 이모티콘, 약어, 불필요 패턴 추가 분석 가능
    # 예시: 자소(ㅋㅋ, ㅜㅜ) 빈도
    print("자소 패턴(ㅋㅋ, ㅎㅎ 등) 빈도:", df['dialogue'].str.count(r'[ㄱ-ㅎㅏ-ㅣ]{2,}').sum())

# 사용 예시
# train_df = pd.read_csv('train.csv')
# basic_eda(train_df)


In [5]:
basic_eda(train_df)

샘플 개수: 12457

[요약문 길이 통계]
count    12457.000000
mean        85.789436
std         33.811948
min         13.000000
25%         61.000000
50%         80.000000
75%        104.000000
max        376.000000
Name: summary_len, dtype: float64
0    Mr. Smith는 Dr. Hawkins에게 건강검진을 받으러 와서, 매년 검진 필...
1    Mrs. Parker가 Ricky와 함께 백신 접종을 위해 방문하였고, Dr. Pe...
2    #Person1#은 열쇠 세트를 잃어버리고 #Person2#에게 찾는 것을 도와달라...
3    #Person1#은 #Person2#가 여자친구가 있고 결혼할 예정이라는 사실을 말...
4    Malik은 Wen과 Nikki에게 춤을 제안하고, Wen은 발을 밟는 것을 감수하...
Name: summary, dtype: object
12452    Tan Ling은 흰머리와 수염이 특징인 Mr. Green을 맞이하여 호텔로 안내합...
12453    #Person1#과 #Person2#는 Mister Ewing의 요청에 따라 회의장...
12454         #Person2#는 #Person1#의 도움으로 5일 동안 소형차를 대여합니다.
12455    #Person2#의 어머니가 직장을 잃으셨다. #Person2#는 어머니가 우울해하...
12456    #Person1#은 다음 주 토요일에 이모부네 가족을 방문하기 위해 짐을 싸야 하는...
Name: summary, dtype: object

[dialogue 길이 통계]
count    12457.000000
mean       406.083487
std        197.566083
min         84.000000
25%        280.000000
50%     

In [3]:
import re
from collections import Counter, defaultdict

# 1. 후보 패턴 리스트 정의
deictic_patterns = [
    r"\b그 사람\b", r"\b이 사람\b", r"\b저 사람\b", r"\b그녀\b", r"\b그\b", r"\b그분\b", r"\b그 분\b",
    r"\b그쪽\b", r"\b이쪽\b", r"\b저쪽\b", r"\b얘\b", r"\b쟤\b", r"\bhe\b", r"\bshe\b",
    r"\b그것\b", r"\b이것\b", r"\b저것\b", r"\b여기\b", r"\b거기\b", r"\b저기\b",
    r"\b그녀는\b", r"\b그는\b", r"\b이이는\b", r"\b저이는\b", r"\b그분은\b", r"\b그 분은\b",
]

# 2. 전체 dialogue에서 패턴별 빈도/예문 추출
def eda_extract_patterns(texts, patterns, n_examples=3):
    pattern_stats = Counter()
    example_dict = defaultdict(list)
    for t in texts:
        for p in patterns:
            matches = re.findall(p, t)
            if matches:
                pattern_stats[p] += len(matches)
                if len(example_dict[p]) < n_examples:
                    example_dict[p].append(t)
    return pattern_stats, example_dict

pattern_stats, example_dict = eda_extract_patterns(train_df['dialogue'], deictic_patterns, n_examples=2)

# 3. 결과 요약 출력
for p in deictic_patterns:
    print(f"\n패턴: {p}\n  빈도: {pattern_stats[p]}")
    for ex in example_dict[p]:
        print(f"  예시: {ex[:100]}...")


패턴: \b그 사람\b
  빈도: 128
  예시: #Person1#: 그 사람 키가 평균 정도였다는 거죠?
#Person2#: 네, 맞아요. 한 5피트 9, 10인치 정도요.
#Person1#: 몸무게는요?
#Person2#: 잘...
  예시: #Person1#: 저 왔어요. 안녕하세요, 아빠. 
#Person2#: 잠깐, 잠깐... 어디 가니? 
#Person1#: 아빠, 엄마한테 벌써 말씀드렸잖아요. 오늘 밤 외출하기...

패턴: \b이 사람\b
  빈도: 6
  예시: #Person1#: 이 모퉁이 쪽 테이블 어때?
#Person2#: 좋아, 여기 앉자.
#Person1#: 아, 근데 너 쟁반에 아무것도 없네.
#Person2#: 응, 그냥 별로...
  예시: #Person1#: 봐봐! 누군가 생일을 축하하고 있어.
#Person2#: 이 사람 21번째 생일인가 보다. 틀림없어.
#Person1#: 왜? 그 사람 알아?
#Person2#...

패턴: \b저 사람\b
  빈도: 32
  예시: #Person1#: 저 왔어요. 안녕하세요, 아빠. 
#Person2#: 잠깐, 잠깐... 어디 가니? 
#Person1#: 아빠, 엄마한테 벌써 말씀드렸잖아요. 오늘 밤 외출하기...
  예시: #Person1#: 얘들아, 나랑 같이 있어. 길 잃으면 안 돼. 
#Person2#: 난 아빠가 아니야. 슈퍼마켓에서 길을 잃을 리 없어.
#Person3#: 엄마, 이제 난 애...

패턴: \b그녀\b
  빈도: 80
  예시: #Person1#: 너 그녀 정말 좋아하는 것 같아, 맞지?
#Person2#: 부인할 수 없어. 첫눈에 반했어. 그녀에 대해 더 알고 싶어.
#Person1#: 근데 듣기로는 그...
  예시: #Person1#: 안녕, Steven. Annie가 너랑 Julia가 싸웠다고 하던데, 맞아?
#Person2#: 맞아. 어제 싸웠어. 그녀가 나한테 저녁 준비 안 했다고 뭐라고...

패턴: \b그\b
  빈도: 4053
  예시: #P

In [2]:
from konlpy.tag import Okt
from collections import Counter
import pandas as pd
from itertools import chain

okt = Okt()

# 모든 대화에서 형태소 분석 (메모리/시간 이슈 있을 수 있으니 샘플링 가능)
def extract_deictic_words(df, pos_list=['Pronoun', 'Determiner', 'Adverb'], save_path=None):
    tokens = chain.from_iterable(okt.pos(d) for d in df['dialogue'])
    # 예시: Pronoun(대명사), Determiner(관형사), Adverb(부사) 중에서 지시어로 쓰이는 품사들
    selected = [word for word, pos in tokens if pos in ['Noun', 'Pronoun', 'Determiner', 'Adverb']]
    counter = Counter(selected)
    df_deictic = pd.DataFrame(counter.most_common(), columns=['word', 'freq'])
    if save_path:
        df_deictic.to_csv(save_path, index=False)
    return df_deictic

# train/dev/test 모두 합치기
all_df = pd.concat([datasets['train'], datasets['dev'], datasets['test']], ignore_index=True)

# 기존 함수에 합친 데이터프레임 전달
deictic_df = extract_deictic_words(all_df, save_path='all_deictic_freq.csv')
print(deictic_df.head(20))

   word   freq
0     네  10294
1     수  10135
2     것   9823
3    정말   8828
4     이   7431
5     그   6732
6     거   6463
7     좀   6351
8     안   6212
9     내   5343
10   우리   5133
11    더   5061
12    해   4951
13   사람   4738
14    너   4635
15   생각   4631
16   여기   4458
17    나   4303
18    일   4107
19    뭐   4068
